![](../assets/placeholder.png)

## Giriş
Bu notebook, Langchain kullanarak ajanlar için araçlar oluşturma sürecinde sizi yönlendirmek için tasarlanmıştır. Araçlar, ajanların ayrılmaz bir parçasıdır ve bir ajanıyla daha fazla yeteneği eklemek ve ajanı bunları faydalı şekillerde birleştirmek için harika bir yoldur. Bu notebook'ta, özel araçlar oluşturmayı ve bunları ajanlarla entegre ederek yeteneklerini geliştirmeyi inceleyeceğiz.

In [ ]:
from dotenv import load_dotenv
import os

# .env dosyasından ortam değişkenlerini yükleyin
load_dotenv()

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=10000)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
tool.description

In [ ]:
tool.args

In [ ]:
tool.run("AI ajanları")

#### Özel araçlar
LangChain'den varsayılan araçları kullanmak zorunda değiliz. Kendi özel araçlarımızı tanımlayabiliriz. Araç, ajanın çağırabileceği açıkça tanımlanmış girdiler ve çıktılara sahip herhangi bir metot olabilir. İki sayıyı bir araya getiren fonksiyon, bir araç olabilir. Özel bir şirket API'sini çağıran fonksiyon, bir araç olabilir. Özel bir LLM'den kod üreten fonksiyon, bir araç olabilir. Bu, bir ajanıyla birçok yeteneği genişletmek için çok sayıda fırsatı açar. Bu bölümde, özel araçları nasıl tanımlayıp ajanlarla entegre edeceğimizi inceleyeceğiz.

In [ ]:
from langchain.tools import tool

# tool dekoratörü yeni bir araç tanımlamanın en kolay yoludur
@tool
def multiply(a: int, b: int) -> int:
    """İki sayıyı çarpın."""
    return a * b

In [ ]:
tool = multiply

In [ ]:
tool.run({'a':65, 'b':67})

### Web retriever aracı
Bir URL'nin içeriğini alabilen bir araç oluşturalım. Bu önceki birkaç notebook'ta birden fazla kez yapmak zorunda kaldığımız bir şeydir. Bu araç, web'den bilgi toplaması gereken ajanlar için özellikle yararlı olacaktır ve soruları yanıtlamak veya görevleri gerçekleştirmek için.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.tools import tool

@tool
def fetch_url_content(url: str) -> str:
    """Bir URL'nin içeriğini almak için bu aracı kullanın"""
    loader = WebBaseLoader(url)
    docs = loader.load()
    return docs

In [ ]:
tool = fetch_url_content
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
content = tool.run({'url': url})
content = str(content)

#### Özeti aracı
Özeti notebook'taunsurlarımızı bir araç haline also getirelim. Bu araç, ajanları, metin içeriğinin kısa özetlerini oluşturmasına olanak tanıyacaktır. Bu, büyük belgeler veya web sayfalarının özünü hızlı bir şekilde anlamak için yararlı olabilir.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool
from langchain_openai import ChatOpenAI

@tool
def summarize_content(content: str) -> str:
    """Metin tabanlı içeriğin özetini oluşturmak için bu aracı kullanın"""
    system_message = "Lütfen içerik için kısa bir özet sağlayın"
    human_message = "İçerik: " + str(content)
    
    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=human_message),
    ]
    model = ChatOpenAI(model="gpt-4o")
    parser = StrOutputParser()
    
    chain = model | parser
    summary = chain.invoke(messages)
    return summary

In [ ]:
tool = summarize_content
tool.run({"content": content})

Bir araçtan çıktıyı diğerine geçirmeyi başardık. Aslında sadece düzenli programlamada fonksiyonlar ile çok basit. İnsanın bunu yapması gerektiğini anlayıp farklı araçlar arasında birçok kombinasyonda kendi başına karar verip görevi tamamlaması güzel olmaz mı? Tam olarak bu AI ajanlarının yaptığı şeydir. Görev tamamlanana kadar bir döngü içinde kendi başlarına araçları çalıştırabilir veya üretebilir. Bunu sonra inceleyeceğiz!